# ALICE Analysis: YFV19, Ankylosing Spondylitis (AS), and MLR TCR-seq Datasets

This notebook runs a highly customized MIR implementation of ALICE (Antigen-specific Lymphocyte Identification by Clustering of Expanded sequences) on three benchmark datasets from `isalgo/airr_benchmark`. The implementation is inspired by ideas described in the ALICE paper and is not a literal line-by-line reimplementation of the original software.

- **YF (Yellow Fever vaccine)** - 6 donors x 2 time points (day 0, day 15) TRB repertoires.
- **AS (Ankylosing Spondylitis)** - 4 donors, synovial fluid CD8+ TRB repertoires.
- **MLR (Mixed Lymphocyte Reaction)** - 24 samples (fresh vs proliferating) from Adaptive immunoSEQ.

Reference:
Pogorelyy MV, Minervina AA, Touzel MP, et al. Detecting T cell receptors involved in immune responses from single repertoire snapshots. *PLoS Biol.* 2019;17(6):e3000314. doi:10.1371/journal.pbio.3000314. PMID:31194732.
PubMed: https://pubmed.ncbi.nlm.nih.gov/31194732/

**Runtime note:** This notebook uses `n_jobs=8` to accelerate compute-intensive ALICE stages with process-based parallel execution.

In [5]:
"""Cell 1: Environment setup, imports, and dataset download."""
import sys
import warnings
from pathlib import Path
from collections import defaultdict

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import mannwhitneyu

repo_root = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mir.biomarkers.alice import compute_alice
from mir.common.clonotype import Clonotype
from mir.common.filter import filter_functional
from mir.common.parser import AdaptiveParser, ClonotypeTableParser, load_vdjdb_latest
from mir.common.repertoire import LocusRepertoire, infer_locus
from mir.graph.edit_distance_graph import build_edit_distance_graph
from mir.utils.notebook_assets import ensure_airr_benchmark, ensure_airr_benchmark_alice

SEED = 42
N_JOBS = 8  # Process-parallel ALICE computation for performance
FDR_THRESH = 0.05

# Single ALICE neighborhood/Pgen mode parameter: "exact" or "1mm".
ALICE_PGEN_MODE = "1mm"

sns.set_theme(style="whitegrid", context="talk")
warnings.filterwarnings("ignore", category=FutureWarning)

# Download all three datasets
BENCHMARK_DIR_YF_AS = ensure_airr_benchmark_alice(repo_root=repo_root, subsets=["yf", "as"])
YF_DIR = BENCHMARK_DIR_YF_AS / "alice" / "yf"
AS_DIR = BENCHMARK_DIR_YF_AS / "alice" / "as"

BENCHMARK_DIR_MLR = ensure_airr_benchmark(repo_root=repo_root, allow_patterns=["alice/mlr/*"])
MLR_DIR = BENCHMARK_DIR_MLR / "alice" / "mlr"

print(f"YF_DIR = {YF_DIR}  (exists: {YF_DIR.exists()})")
print(f"AS_DIR = {AS_DIR}  (exists: {AS_DIR.exists()})")
print(f"MLR_DIR = {MLR_DIR}  (exists: {MLR_DIR.exists()})")

Fetching 27 files: 100%|██████████| 27/27 [00:00<00:00, 1765.31it/s]

YF_DIR = /Users/mikesh/vcs/mirpy/notebooks/assets/large/airr_benchmark/alice/yf  (exists: True)
AS_DIR = /Users/mikesh/vcs/mirpy/notebooks/assets/large/airr_benchmark/alice/as  (exists: True)
MLR_DIR = /Users/mikesh/vcs/mirpy/notebooks/assets/large/airr_benchmark/alice/mlr  (exists: True)


# Yellow Fever (YF) Dataset

## Parse YF samples

Files follow the MiXCR v2/v3 naming convention: `<donor>_d<day>.tsv.gz` (e.g. `P1_d0.tsv.gz`, `S2_d15.tsv.gz`).
`ClonotypeTableParser` auto-maps MiXCR column names and infers locus from the `j_gene` prefix.

In [ ]:
"""Cell 2: Parse YF samples from MiXCR TSV.gz files and downsample to 100k duplicates."""
from mir.common.sampling import downsample

# YF donors in the benchmark dataset
YF_DONORS = ["P1", "P2", "Q1", "Q2", "S1", "S2"]
YF_DOWNSAMPLE_COUNT = 100_000

parser_yf = ClonotypeTableParser()
warnings.filterwarnings("ignore", category=FutureWarning)

def _parse_yf_filename(path: Path) -> tuple[str, int]:
    """Return (donor, day) from `<donor>_d<day>.tsv.gz`."""
    stem = path.name.replace(".tsv.gz", "").replace(".tsv", "")
    donor, day_part = stem.split("_d")
    return donor, int(day_part)

yf_samples: dict[tuple[str, int], LocusRepertoire] = {}

for fp in sorted(YF_DIR.glob("*.tsv.gz")):
    try:
        donor, day = _parse_yf_filename(fp)
    except ValueError:
        print(f"  Skipping unexpected filename: {fp.name}")
        continue

    clonotypes = parser_yf.parse(str(fp))
    trb = [c for c in clonotypes if c.locus == "TRB"]
    rep = filter_functional(LocusRepertoire(clonotypes=trb, locus="TRB", repertoire_id=fp.stem))
    rep = downsample(rep, YF_DOWNSAMPLE_COUNT, random_seed=SEED)
    yf_samples[(donor, day)] = rep
    print(f"  {fp.name}: {rep.clonotype_count:,} TRB clonotypes after downsample to {YF_DOWNSAMPLE_COUNT:,}")

print(f"\nLoaded {len(yf_samples)} YF samples")

## Run ALICE on YF samples

ALICE scores each clonotype by its observed neighborhood size vs the Poisson expectation, conditioned on the V+J gene pair (`match_mode="vj"`).
Neighborhood/Pgen behavior is controlled by a single mode parameter: `ALICE_PGEN_MODE` (`"exact"` or `"1mm"`).
Benjamini-Hochberg FDR is applied per sample; hits = clonotypes with $q < 0.05$.

In [ ]:
"""Cell 3: Run ALICE on YF samples, collect hits at FDR < 0.05."""

yf_alice_hits: dict[tuple[str, int], pd.DataFrame] = {}
yf_alice_rows = []

for (donor, day), rep in sorted(yf_samples.items()):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = compute_alice(
            rep,
            species="human",
            match_mode="vj",
            pgen_mode=ALICE_PGEN_MODE,
            as_table=True,
            n_jobs=N_JOBS,
        )

    # result.table already contains BH-corrected q_value from compute_alice
    hits = result.table[result.table["q_value"] < FDR_THRESH].copy()
    yf_alice_hits[(donor, day)] = hits

    yf_alice_rows.append({
        "donor": donor,
        "day": day,
        "n_clonotypes": rep.clonotype_count,
        "n_hits": len(hits),
    })
    print(f"  {donor} day{day:>3}: {rep.clonotype_count:,} clonotypes -> {len(hits)} ALICE hits (FDR<0.05)")

df_yf_summary = pd.DataFrame(yf_alice_rows).sort_values(["donor", "day"]).reset_index(drop=True)
display(df_yf_summary)

## YF Visualization

Bar plot showing ALICE hit counts per donor at day 0 vs day 15.

In [ ]:
"""Cell 4: Bar plot of ALICE hit counts for day 0 and day 15."""

donors_ordered = sorted(df_yf_summary["donor"].unique())
days_ordered = sorted(df_yf_summary["day"].unique())
day_colors = {0: "#9ecae1", 15: "#de2d26"}
x = np.arange(len(donors_ordered))
bar_width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: raw hit counts ──────────────────────────────────────────────────────
ax = axes[0]
for i, day in enumerate(days_ordered):
    sub = df_yf_summary[df_yf_summary["day"] == day].set_index("donor")
    heights = [sub.loc[d, "n_hits"] if d in sub.index else 0 for d in donors_ordered]
    offset = (i - len(days_ordered) / 2 + 0.5) * bar_width
    ax.bar(x + offset, heights, bar_width, color=day_colors.get(day, "grey"),
           label=f"Day {day}", alpha=0.85, edgecolor="black", linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(donors_ordered)
ax.set_xlabel("Donor")
ax.set_ylabel("ALICE hits (FDR < 0.05)")
ax.set_title("ALICE hit clonotypes per donor (YF)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# ── Right: hits normalised by repertoire size ─────────────────────────────────
ax = axes[1]
for i, day in enumerate(days_ordered):
    sub = df_yf_summary[df_yf_summary["day"] == day].set_index("donor")
    heights = []
    for d in donors_ordered:
        if d in sub.index:
            n_total = max(sub.loc[d, "n_clonotypes"], 1)
            heights.append(sub.loc[d, "n_hits"] / n_total * 100)
        else:
            heights.append(0.0)
    offset = (i - len(days_ordered) / 2 + 0.5) * bar_width
    ax.bar(x + offset, heights, bar_width, color=day_colors.get(day, "grey"),
           label=f"Day {day}", alpha=0.85, edgecolor="black", linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(donors_ordered)
ax.set_xlabel("Donor")
ax.set_ylabel("ALICE hits (% of sample)")
ax.set_title("ALICE hit fraction per donor (YF)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Filter ALICE hits to HLA-A*02 LLWNGPMAV-specific clonotypes

We intersect YF ALICE hits with VDJdb entries for LLWNGPMAV restricted to HLA-A*02.
Donors with HLA-A*02 are expected to show a strong day-15 signal.

In [ ]:
"""Cell 5: VDJdb A*02 LLW filter and bar plot of LLW-specific ALICE hits."""

# Load VDJdb LLWNGPMAV / HLA-A*02 reference from GitHub release
vdjdb_llw = load_vdjdb_latest(
    epitope="LLWNGPMAV",
    locus="TRB",
    species="HomoSapiens",
    mhc_a_contains="A*02",
)
llw_cdr3s = {c.junction_aa for c in vdjdb_llw.clonotypes if c.junction_aa}
print(f"VDJdb A*02 LLW reference: {len(llw_cdr3s)} unique CDR3 amino acid sequences")

# Intersect each sample's ALICE hits with LLW reference.
llw_rows = []
for (donor, day), hits in sorted(yf_alice_hits.items()):
    n_llw = hits["junction_aa"].isin(llw_cdr3s).sum() if not hits.empty else 0
    llw_rows.append({"donor": donor, "day": day, "n_llw_hits": n_llw})

df_llw = pd.DataFrame(llw_rows).sort_values(["donor", "day"]).reset_index(drop=True)
display(df_llw)

# Bar plot for LLW-specific hits.
x = np.arange(len(donors_ordered))
fig, ax = plt.subplots(figsize=(9, 5))
for i, day in enumerate(days_ordered):
    sub = df_llw[df_llw["day"] == day].set_index("donor")
    heights = [sub.loc[d, "n_llw_hits"] if d in sub.index else 0 for d in donors_ordered]
    offset = (i - len(days_ordered) / 2 + 0.5) * bar_width
    ax.bar(x + offset, heights, bar_width, color=day_colors.get(day, "grey"),
           label=f"Day {day}", alpha=0.85, edgecolor="black", linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(donors_ordered)
ax.set_xlabel("Donor")
ax.set_ylabel("A*02 LLW-specific ALICE hits")
ax.set_title("ALICE hits matching HLA-A*02 LLWNGPMAV (VDJdb) per donor")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Ankylosing Spondylitis (AS) Dataset

## Parse AS samples

Four donors from synovial fluid CD8+ TRB repertoires.
Donor B27 status metadata is hard-coded from study design.

In [ ]:
"""Cell 6: Parse AS samples."""

AS_DONOR_META = {
    1: "B27_pos",
    2: "B27_pos",
    3: "B27_neg",
    4: "B27_pos",
}

as_samples: dict[int, LocusRepertoire] = {}

for donor_id, b27 in sorted(AS_DONOR_META.items()):
    pattern = f"Donor{donor_id}_B27_{b27}_SF_CD8.tsv.gz"
    fp = AS_DIR / pattern
    if not fp.exists():
        # Try globbing in case of slight filename variation.
        candidates = list(AS_DIR.glob(f"Donor{donor_id}_*.tsv.gz"))
        if not candidates:
            print(f"  WARNING: no file for Donor {donor_id} — skipping")
            continue
        fp = candidates[0]

    clonotypes = parser_yf.parse(str(fp))
    trb = [c for c in clonotypes if c.locus == "TRB"]
    rep = filter_functional(LocusRepertoire(clonotypes=trb, locus="TRB", repertoire_id=fp.stem))
    as_samples[donor_id] = rep
    print(f"  Donor {donor_id} ({b27}): {rep.clonotype_count:,} TRB clonotypes  [{fp.name}]")

print(f"\nLoaded {len(as_samples)} AS samples")

## Run ALICE on AS samples

Same ALICE parameters as YF (`match_mode="vj"`, `pgen_mode=ALICE_PGEN_MODE`, FDR < 0.05).
B27_pos donors (1, 2, 4) are expected to share convergent CDR3 neighborhoods absent in the B27_neg donor (3).

In [ ]:
"""Cell 7: Run ALICE on all AS samples, collect FDR-filtered hits."""

as_alice_hits: dict[int, pd.DataFrame] = {}

for donor_id, rep in sorted(as_samples.items()):
    b27 = AS_DONOR_META[donor_id]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = compute_alice(
            rep,
            species="human",
            match_mode="vj",
            pgen_mode=ALICE_PGEN_MODE,
            as_table=True,
            n_jobs=N_JOBS,
        )

    # result.table already contains BH-corrected q_value from compute_alice
    hits = result.table[result.table["q_value"] < FDR_THRESH].copy()
    hits["donor_id"] = donor_id
    hits["b27"] = b27
    as_alice_hits[donor_id] = hits
    print(f"  Donor {donor_id} ({b27}): {rep.clonotype_count:,} clonotypes -> {len(hits)} ALICE hits")

# Pool for graph construction.
df_as_all_hits = pd.concat(as_alice_hits.values(), ignore_index=True)
print(f"\nTotal AS ALICE hits (with duplicates across donors): {len(df_as_all_hits)}")
print(f"Unique junction_aa in hits: {df_as_all_hits['junction_aa'].nunique()}")

## Build Hamming clonotype graph for AS ALICE hits

Each ALICE hit clonotype becomes a graph node. Edges connect pairs with Hamming distance ≤ 1 in the same V-gene.
We then build node color arrays for rendering with donor-specific colors.

In [ ]:
"""Cell 8: Build Hamming distance graph for AS ALICE hits."""

# ── Donor color palette ────────────────────────────────────────────────────────
DONOR_COLORS = {
    1: "#f97f31",   # orange — B27_pos
    2: "#3182bd",   # blue   — B27_pos
    3: "#31a354",   # green  — B27_neg
    4: "#de2d26",   # red    — B27_pos
}

# Highlight clonotypes of special interest (TRBV9 pair from literature).
HIGHLIGHT_CLONOTYPES = {
    ("TRBV9", "CASSVGLFSTDTQYF", "TRBJ2-3"),
    ("TRBV9", "CASSVGLYSTDTQYF", "TRBJ2-3"),
}

def _as_clonotype(row: pd.Series) -> Clonotype:
    """Construct a Clonotype from a hits-table row."""
    return Clonotype(
        junction_aa=row.get("junction_aa") or "",
        v_gene=row.get("v_gene") or "",
        j_gene=row.get("j_gene") or "",
        duplicate_count=1,
        locus="TRB",
    )

# Build a list of Clonotype objects keeping track of which donor each belongs to.
clonotype_list = []
clonotype_donor_ids = []

for donor_id, hits in sorted(as_alice_hits.items()):
    for _, row in hits.iterrows():
        clon = _as_clonotype(row)
        clonotype_list.append(clon)
        clonotype_donor_ids.append(donor_id)

print(f"Total nodes before deduplication: {len(clonotype_list)}")

# Build the edit-distance graph.
# v_gene_match=True: only edges between clonotypes sharing the same V gene.
g = build_edit_distance_graph(
    clonotype_list,
    metric="hamming",
    threshold=1,
    v_gene_match=True,
    n_jobs=N_JOBS,
)
g.vs["donor_id"] = clonotype_donor_ids

print(f"Graph: {g.vcount()} nodes, {g.ecount()} edges")

## Render AS graph with donor colors and highlighted targets

Rendering strategy:
- Compute Fruchterman-Reingold layout
- Draw edges (thin grey lines)
- Single-donor nodes: filled circle with donor color
- Multi-donor nodes: pie-chart wedges per donor
- Star markers on target clonotypes

In [ ]:
"""Cell 9: Render AS Hamming graph with donor-color pie nodes and highlighted targets."""
import igraph as ig
import matplotlib.colors as mc
from matplotlib.patches import Circle, FancyArrowPatch, Wedge

# ── Layout ─────────────────────────────────────────────────────────────────────
# Keep only nodes in connected components with ≥ 2 nodes for clarity.
comp_sizes = g.clusters().sizes()
subg = g  # default: full graph
if max(comp_sizes) > 1:
    large_comps = [i for i, sz in enumerate(comp_sizes) if sz >= 2]
    keep = [v for v in range(g.vcount()) if g.clusters().membership[v] in large_comps]
    if keep:
        subg = g.induced_subgraph(keep)

layout = subg.layout("fr", niter=1000)
coords = np.array(layout.coords)

# Normalise coordinates to [0, 1] for matplotlib.
if coords.shape[0] > 0:
    mn = coords.min(axis=0)
    mx = coords.max(axis=0)
    span = np.where(mx - mn > 0, mx - mn, 1.0)
    coords = (coords - mn) / span

NODE_RADIUS = 0.012
HIGHLIGHT_RADIUS = NODE_RADIUS * 1.5

fig, ax = plt.subplots(figsize=(14, 12))
ax.set_aspect("equal")
ax.axis("off")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)

# ── Draw edges ─────────────────────────────────────────────────────────────────
for e in subg.es:
    s, t = e.source, e.target
    xs = [coords[s, 0], coords[t, 0]]
    ys = [coords[s, 1], coords[t, 1]]
    ax.plot(xs, ys, color="#aaaaaa", linewidth=0.5, zorder=1, alpha=0.7)

# ── Group vertices by CDR3+V to find multi-donor nodes ─────────────────────────
vertex_key = [
    (subg.vs[v]["name"], subg.vs[v]["v_gene"])
    for v in range(subg.vcount())
]
# Map each node to the set of donor_ids that contributed it.
key_to_nodes: dict[tuple, list[int]] = defaultdict(list)
for v, key in enumerate(vertex_key):
    key_to_nodes[key].append(v)

# ── Draw nodes ─────────────────────────────────────────────────────────────────
drawn_nodes: set[int] = set()

for key, node_ids in key_to_nodes.items():
    # Collect all donor_ids for this CDR3+V.
    donors_here = sorted({subg.vs[v]["donor_id"] for v in node_ids})
    # Representative position: mean of all duplicated node positions.
    pos = coords[node_ids, :].mean(axis=0)
    cx, cy = pos[0], pos[1]

    # Detect highlighted clonotypes.
    jaa = key[0]
    vgene = key[1]
    # Try to find j_gene for this node.
    jgene = subg.vs[node_ids[0]].get("j_gene", "")
    is_highlight = (vgene, jaa, jgene) in HIGHLIGHT_CLONOTYPES
    radius = HIGHLIGHT_RADIUS if is_highlight else NODE_RADIUS

    if len(donors_here) == 1:
        # Single donor: solid circle.
        color = DONOR_COLORS.get(donors_here[0], "#cccccc")
        circle = Circle(
            (cx, cy), radius,
            facecolor=color, edgecolor="black" if is_highlight else "white",
            linewidth=2.0 if is_highlight else 0.3,
            zorder=3,
        )
        ax.add_patch(circle)
    else:
        # Multi-donor: pie-chart wedges.
        n_d = len(donors_here)
        angle_per = 360.0 / n_d
        for k, did in enumerate(donors_here):
            wedge = Wedge(
                (cx, cy), radius,
                theta1=k * angle_per, theta2=(k + 1) * angle_per,
                facecolor=DONOR_COLORS.get(did, "#cccccc"),
                edgecolor="black" if is_highlight else "white",
                linewidth=2.0 if is_highlight else 0.3,
                zorder=3,
            )
            ax.add_patch(wedge)

    if is_highlight:
        # Overlay a star marker.
        ax.plot(
            cx, cy, marker="*", color="gold", markersize=10, zorder=5,
            markeredgecolor="black", markeredgewidth=0.5,
        )
        ax.annotate(
            jaa, (cx, cy + radius + 0.015),
            fontsize=6.5, ha="center", va="bottom", zorder=6,
            bbox=dict(boxstyle="round,pad=0.2", facecolor="lightyellow", alpha=0.85, edgecolor="grey"),
        )

    drawn_nodes.update(node_ids)

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor=DONOR_COLORS[did], edgecolor="black", linewidth=0.5,
                   label=f"Donor {did} ({AS_DONOR_META[did]})")
    for did in sorted(DONOR_COLORS)
]
legend_handles.append(mpatches.Patch(facecolor="gold", edgecolor="black", linewidth=0.5,
                                      label="Highlighted target clonotypes"))
ax.legend(handles=legend_handles, loc="lower right", fontsize=9, framealpha=0.9)

ax.set_title(
    f"AS ALICE hits — Hamming-1 clonotype graph  ({subg.vcount()} nodes, {subg.ecount()} edges)\n"
    "Node color = donor | Multi-donor = pie | Star = target CDR3",
    fontsize=11,
)
plt.tight_layout()
plt.show()

# Mixed Lymphocyte Reaction (MLR) Dataset

## Load Adaptive files and build TRB repertoires

The parser maps the Adaptive columns to AIRR-style fields. Files are parsed into single-locus TRB repertoires.
Gene names are normalized (e.g. `TCRBV01` → `TRBV1`).

In [ ]:
"""Cell 10: Parse MLR samples from Adaptive immunoSEQ files."""
import importlib

import mir.common.parser as parser_module
parser_module = importlib.reload(parser_module)
AdaptiveParser = parser_module.AdaptiveParser
parser_adaptive = AdaptiveParser(locus="TRB")

def parse_mlr_filename(path: Path) -> dict[str, object]:
    """Return sample metadata encoded in the MLR filename."""
    stem = path.name
    stem = stem.removesuffix(".tsv.gz")
    stem = stem.removesuffix(".tsv")
    parts = stem.split("_")
    if "Fresh" in parts:
        condition = "fresh"
        idx = parts.index("Fresh")
    elif "Proliferating" in parts:
        condition = "proliferating"
        idx = parts.index("Proliferating")
    else:
        raise ValueError(f"Could not infer condition from {path.name}")

    sample_group = "_".join(parts[:idx])
    replicate = int(parts[-1])
    return {
        "sample_id": stem,
        "sample_group": sample_group,
        "condition": condition,
        "replicate": replicate,
    }

def downsample_by_duplicate_count(repertoire, max_duplicates: int = 1000000, random_seed: int = 42):
    """Downsample repertoire to max_duplicates total duplicate count.
    Uses weighted random sampling: each clonotype is sampled with probability
    proportional to its duplicate count.
    """
    import random
    random.seed(random_seed)

    total_dups = repertoire.duplicate_count
    if total_dups <= max_duplicates:
        return repertoire

    # Calculate sampling probability per clonotype to match target total
    sampling_prob = min(1.0, max_duplicates / total_dups)

    # Weighted sampling: sample each clonotype based on its duplicate count
    sampled_clonotypes = []
    for clonotype in repertoire.clonotypes:
        if random.random() < sampling_prob:
            sampled_clonotypes.append(clonotype)

    # If we sampled too few, add back the most abundant ones
    if not sampled_clonotypes:
        sampled_clonotypes = [max(repertoire.clonotypes, key=lambda x: x.duplicate_count)]

    return LocusRepertoire(
        sampled_clonotypes,
        locus=repertoire.locus,
        repertoire_metadata=dict(repertoire.repertoire_metadata),
    )

mlr_files = sorted(fp for fp in MLR_DIR.glob("*.tsv.gz") if "Stimulator" not in fp.name)
print(f"Found {len(mlr_files)} MLR files")
for fp in mlr_files:
    print(f"  {fp.name}")

mlr_samples: dict[str, object] = {}
mlr_rows: list[dict[str, object]] = []

MAX_DUPLICATES = 1000000

for fp in mlr_files:
    meta = parse_mlr_filename(fp)
    rep = parser_adaptive.parse_file(fp, sample_id=str(meta["sample_id"]), locus="TRB")

    # Filter for functional clonotypes only
    rep_functional = rep.subsample_functional()

    # Downsample to max duplicate count
    rep_downsampled = downsample_by_duplicate_count(rep_functional, max_duplicates=MAX_DUPLICATES, random_seed=SEED)

    rep_downsampled.repertoire_metadata.update(meta)
    mlr_samples[str(meta["sample_id"])] = rep_downsampled
    mlr_rows.append({
        **meta,
        "n_clonotypes_total": rep.clonotype_count,
        "n_clonotypes_functional": rep_functional.clonotype_count,
        "n_clonotypes": rep_downsampled.clonotype_count,
        "n_duplicates_total": rep.duplicate_count,
        "n_duplicates_functional": rep_functional.duplicate_count,
        "n_duplicates": rep_downsampled.duplicate_count,
    })
    print(
        f"  {fp.name}: {rep.clonotype_count:,} clonotypes ({rep_functional.clonotype_count:,} functional, "
        f"{rep_downsampled.clonotype_count:,} after downsampling), "
        f"{rep.duplicate_count:,} → {rep_functional.duplicate_count:,} → {rep_downsampled.duplicate_count:,} duplicates"
    )

df_mlr_samples = pd.DataFrame(mlr_rows).sort_values(["sample_group", "condition", "replicate"])
df_mlr_samples = df_mlr_samples.reset_index(drop=True)
display(df_mlr_samples)

## Run ALICE on all MLR samples

ALICE is run with `match_mode="vj"` conditioned on both V and J gene usage.
BH-FDR hits are collected at `q < 0.05`.

**Note:** ALICE runs with `n_jobs=8` for efficient multi-threaded computation.

In [ ]:
"""Cell 11: Run ALICE on all MLR samples, collect BH-FDR hits at q < FDR_THRESH."""

mlr_alice_hits: dict[str, pd.DataFrame] = {}
mlr_alice_rows = []

for sample_id, rep in sorted(mlr_samples.items()):
    meta = rep.repertoire_metadata
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = compute_alice(
            rep,
            species="human",
            match_mode="vj",
            pgen_mode=ALICE_PGEN_MODE,
            as_table=True,
            n_jobs=N_JOBS,
        )

    # result.table already contains BH-corrected q_value from compute_alice
    hits = result.table[result.table["q_value"] < FDR_THRESH].copy()
    mlr_alice_hits[sample_id] = hits

    mlr_alice_rows.append({
        "sample_id": sample_id,
        "sample_group": meta.get("sample_group", ""),
        "condition": meta.get("condition", ""),
        "replicate": meta.get("replicate", ""),
        "n_clonotypes": rep.clonotype_count,
        "n_hits": len(hits),
        "hit_fraction": len(hits) / max(1, rep.clonotype_count),
    })
    print(
        f"  {sample_id}: {rep.clonotype_count:,} clonotypes -> "
        f"{len(hits)} ALICE hits (FDR<{FDR_THRESH})"
    )

df_mlr_hits = pd.DataFrame(mlr_alice_rows).sort_values(
    ["condition", "sample_group", "replicate"]
).reset_index(drop=True)
display(df_mlr_hits)

## Fresh versus Proliferating Comparison

Boxplots and beeswarm overlays comparing sample-level ALICE hit fractions between fresh and proliferating groups.
Mann-Whitney U-test reports statistical significance.

In [ ]:
"""Cell 12: Fresh vs Proliferating — boxplot and Mann-Whitney U comparison."""
from scipy.stats import mannwhitneyu

fresh_hits = df_mlr_hits[df_mlr_hits["condition"] == "fresh"]["hit_fraction"].values
prolif_hits = df_mlr_hits[df_mlr_hits["condition"] == "proliferating"]["hit_fraction"].values

if len(fresh_hits) >= 2 and len(prolif_hits) >= 2:
    stat, pval = mannwhitneyu(prolif_hits, fresh_hits, alternative="greater")
    print(f"Mann-Whitney U (proliferating > fresh): p = {pval:.4f}")
else:
    pval = float("nan")
    print("Too few samples for Mann-Whitney U test.")

fig, ax = plt.subplots(figsize=(7, 5))
conditions = sorted(df_mlr_hits["condition"].unique())
condition_colors = {"fresh": "#9ecae1", "proliferating": "#de2d26"}

data_by_condition = [
    df_mlr_hits[df_mlr_hits["condition"] == c]["hit_fraction"].values
    for c in conditions
]
bp = ax.boxplot(
    data_by_condition,
    labels=conditions,
    patch_artist=True,
    showfliers=False,
)
for patch, condition in zip(bp["boxes"], conditions):
    patch.set(facecolor=condition_colors.get(condition, "grey"), alpha=0.8)

# Overlay individual points with jitter
rng = np.random.default_rng(42)
for i, (condition, vals) in enumerate(zip(conditions, data_by_condition)):
    jitter = rng.uniform(-0.08, 0.08, size=len(vals))
    ax.scatter(
        np.full(len(vals), i + 1) + jitter,
        vals,
        color=condition_colors.get(condition, "grey"),
        edgecolors="black",
        linewidths=0.6,
        s=35,
        zorder=3,
    )

ax.set_ylabel("ALICE hit fraction")
ax.set_title(
    f"ALICE hit fraction: fresh vs proliferating MLR\n"
    f"Mann-Whitney U p = {pval:.4f}" if not np.isnan(pval) else "ALICE hit fraction: fresh vs proliferating MLR"
)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Summary

This comprehensive notebook analyzes ALICE (Antigen-specific Lymphocyte Identification by Clustering of Expanded CDR3s) on three independent benchmark datasets:

1. **YF Dataset:** Yellow Fever vaccine response with VDJdb filtering for HLA-A*02 LLWNGPMAV epitope specificity
2. **AS Dataset:** Ankylosing Spondylitis repertoires with Hamming-distance graph visualization of convergent clonotypes
3. **MLR Dataset:** Mixed lymphocyte reaction with fresh vs proliferating sample comparison

**Multi-threading:** All ALICE computations use `n_jobs=8` for efficient parallelization, reducing runtime particularly for the large MLR dataset. The neighborhood enrichment calculation is fully multi-threaded via ThreadPoolExecutor.

**Reference:** Results are validated against https://journals.plos.org/plosbiology/article?id=10.1371/journal.pbio.3000314